In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from collections import defaultdict

df = pd.read_csv('/content/drive/MyDrive/NguyenTheLuan_23520899/Data/experiment_data/GSE131907_Lung_Cancer_Feature_Summary.csv')
sample_to_stage = dict(zip(df['Samples'], df['Stages']))

In [ ]:
with open("/content/drive/MyDrive/NguyenTheLuan_23520899/Data/experiment_data/GSE131907_Lung_Cancer_normalized_log2TPM_matrix.txt", 'r') as f:
    header = f.readline().strip().split('\t')
    gene_col = header[0]
    sample_cols = header[1:]

target_per_stage = 6250
stage_counts = defaultdict(int)
selected_samples = []
selected_stages = {}

for full_sample in sample_cols:
    parts = full_sample.split('_')
    if len(parts) < 3:
        continue
    short_sample = f"{parts[-2]}_{parts[-1]}"

    stage = sample_to_stage.get(short_sample)
    if stage and stage_counts[stage] < target_per_stage:
        selected_samples.append(full_sample)
        selected_stages[full_sample] = stage
        stage_counts[stage] += 1

    if all(count >= target_per_stage for count in stage_counts.values()):
        break

print(stage_counts)


defaultdict(<class 'int'>, {'IA3': 6250, 'IV': 6250, 'IA': 6250, 'IA2': 3092, 'IIA': 6250, 'IIIA': 6250, 'IB': 6250, 'IIB': 6250})


In [ ]:
usecols = [gene_col] + selected_samples
df = pd.read_csv("/content/drive/MyDrive/NguyenTheLuan_23520899/Data/experiment_data/GSE131907_Lung_Cancer_normalized_log2TPM_matrix.txt", sep='\t', usecols=usecols)

df_T = df.set_index(gene_col).T
df_T.index.name = "Sample"

df_T["Stage"] = df_T.index.map(selected_stages)

df_T.to_csv("annotated_data_12500_samples_per_class.csv", index=True)